In [4]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

options = Options()
# options.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def crawl_naver_reviews(url):
    driver.get(url)

    # 1. iframe 전환 (PC 지도 URL일 경우를 대비)
    time.sleep(3)
    try:
        driver.switch_to.frame("entryIframe")
    except:
        pass # 모바일 URL이거나 iframe이 없으면 그냥 넘어갑니다.

    # 2. 리뷰 데이터가 화면에 뜰 때까지 대기 (최대 10초)
    print("페이지 로딩을 기다리는 중입니다...")
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "li.place_apply_pui"))
        )
        print("리뷰 로딩 완료! [더보기] 버튼 클릭을 시작합니다.")
    except Exception as e:
        print("10초가 지나도 리뷰를 찾지 못했습니다. URL이나 화면 상태를 확인해주세요.")
        return []

    # 3. "펼쳐서 더보기" 버튼 끝까지 클릭
    click_count = 0
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, "a.fvwqf")
            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(1.5) # 클릭 후 새 리뷰가 뜰 때까지 대기
            click_count += 1
        except:
            print(f"더보기 클릭 완료! (총 {click_count}번 클릭)")
            break

    # 4. 데이터 수집 (안전한 방식 적용)
    review_data = []
    items = driver.find_elements(By.CSS_SELECTOR, "li.place_apply_pui")
    print(f"화면에서 총 {len(items)}개의 리뷰 덩어리를 찾았습니다. 추출을 시작합니다.")

    for item in items:
        # try-except 대신 find_elements를 사용하여 항목이 없어도 튕기지 않게 방어
        id_elements = item.find_elements(By.CSS_SELECTOR, ".pui__NMi-Dp")
        user_id = id_elements[0].text if id_elements else "아이디 없음"

        date_elements = item.find_elements(By.CSS_SELECTOR, ".pui__QKE5Pr time")
        date = date_elements[0].text if date_elements else "날짜 없음"

        content_elements = item.find_elements(By.CSS_SELECTOR, ".pui__vn15t2 a")
        content = content_elements[0].get_attribute("textContent").strip() if content_elements else "내용 없음"

        # 내용이 아예 없는 빈 껍데기 리뷰는 제외
        if user_id != "아이디 없음" or content != "내용 없음":
            review_data.append({
                "작성일자": date,
                "아이디": user_id,
                "리뷰내용": content
            })

    return review_data

try:
    # URL에 실제로 테스트하시는 네이버 리뷰 주소를 넣어주세요.
    url = "https://map.naver.com/p/entry/place/1740956299?lng=127.4480068&lat=37.8134677&placePath=/review?additionalHeight=76&businessCategory=camping&fromPanelNum=1&locale=ko&svcName=map_pcv5&timestamp=202602231735&reviewSort=recent&additionalHeight=76&entry=plt&searchType=place&c=14.85,0,0,0,dh"

    results = crawl_naver_reviews(url)

    if len(results) > 0:
        df = pd.DataFrame(results)
        df.to_excel("naver_reviews.xlsx", index=False)
        print(f"성공! 총 {len(df)}개의 리뷰가 'naver_reviews.xlsx'에 저장되었습니다.")
    else:
        print("수집된 데이터가 없어 엑셀 파일을 생성하지 않았습니다.")

finally:
    driver.quit()

페이지 로딩을 기다리는 중입니다...
리뷰 로딩 완료! [더보기] 버튼 클릭을 시작합니다.
더보기 클릭 완료! (총 187번 클릭)
화면에서 총 1880개의 리뷰 덩어리를 찾았습니다. 추출을 시작합니다.
성공! 총 1880개의 리뷰가 'naver_reviews.xlsx'에 저장되었습니다.
